# Travel Radar Data and AI Team LLM & GenAI Basics


Welcome to this interactive notebook! You'll learn:
- ✅ How to set up and authenticate with OpenRouter API
- ✅ How to work with different LLM models
- ✅ Prompt engineering fundamentals
- ✅ Parameter tuning and configuration
- ✅ Error handling and best practices
- ✅ Real-world use cases and applications


**Prerequisites:** Basic Python knowledge

---

## Section 1: Setup and Installation 🔧

Before we start, we need to install the required libraries for interacting with the OpenRouter API.

### What you'll install:
- **openai**: Python SDK to interact with OpenAI-compatible APIs (like OpenRouter)
- **python-dotenv**: To manage environment variables securely
- **requests**: For making HTTP requests
- **json**: For handling JSON data (comes with Python)

In [ ]:
# Install required libraries
# Run this cell only once!

import subprocess
import sys

libraries = [
    'openai',
    'python-dotenv',
    'requests'
]

for lib in libraries:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', lib, '-q'])
        print(f"✓ {lib} installed successfully")
    except Exception as e:
        print(f"✗ Error installing {lib}: {e}")

print("\n✅ All libraries installed!")

### Setting Up Your API Key

You'll need an OpenRouter API key to get started. Here's how:

1. **Get your API key:**
   - Visit [OpenRouter.ai](https://openrouter.ai)
   - Sign up for a free account
   - Go to Settings → API Keys
   - Copy your API key

2. **Set it as an environment variable:**
   - The code below will help you set it securely

In [ ]:
import os
from getpass import getpass

# METHOD 1: Set API key from environment variable
# If you've already set OPENROUTER_API_KEY in your system
api_key_env = 'os.getenv('OPENROUTER_API_KEY')'

if api_key_env:
    api_key = api_key_env
    print("✓ API key loaded from environment variable")
else:
    # METHOD 2: Prompt user to enter API key
    print("No API key found in environment.")
    print("Enter your OpenRouter API key below (it won't be displayed):")
    api_key = getpass("OpenRouter API Key: ")

    # Verify the key format
    if api_key.startswith('sk-or-'):
        print("✓ API key format looks correct!")
    else:
        print("⚠️ Warning: API key doesn't start with 'sk-or-'. Check if it's correct.")

# Store in environment for this session
os.environ['OPENROUTER_API_KEY'] = api_key

print(f"\nAPI Key Set: {api_key[:10]}...{api_key[-5:]}")

---

## Section 2: Importing Required Libraries 📚

Let's import all the libraries we'll use throughout this notebook.

In [ ]:
# Core libraries for API interaction
import requests
import json
import os
import time
from datetime import datetime

# For structured data handling
import pandas as pd

# For environment variable management
from dotenv import load_dotenv

# Load environment variables from .env file (if it exists)
load_dotenv()

print("✓ All libraries imported successfully!")
print("\nLoaded libraries:")
print("  • requests - HTTP requests")
print("  • json - JSON data handling")
print("  • os - Operating system interface")
print("  • time - Time-related functions")
print("  • pandas - Data analysis")
print("  • dotenv - Environment variable management")

---

## Section 3: Understanding the OpenRouter API 🔬

### What is OpenRouter?
OpenRouter is an API gateway that provides access to multiple LLM models through a single interface.

### Key Concepts:

**1. Endpoints:**
- `/chat/completions` - For chat-based interactions (most common)
- `/models` - List available models
- `/auth/key` - Check API key status

**2. Authentication:**
- API key in `Authorization` header
- Format: `Bearer YOUR_API_KEY`

**3. Request Structure:**
```json
{
  "model": "openai/gpt-4",
  "messages": [
    {"role": "user", "content": "Hello!"}
  ],
  "temperature": 0.7,
  "max_tokens": 100
}
```

**4. Response Structure:**
```json
{
  "choices": [
    {
      "message": {
        "role": "assistant",
        "content": "Response text here"
      }
    }
  ],
  "usage": {
    "prompt_tokens": 10,
    "completion_tokens": 20
  }
}
```

In [ ]:
# Helper function to make API calls to OpenRouter

def call_openrouter_api(model, messages, temperature=0.7, max_tokens=500):
    """
    Call OpenRouter API with specified parameters.

    Args:
        model (str): Model ID (e.g., 'openai/gpt-4', 'anthropic/claude-3-opus')
        messages (list): List of message dictionaries with 'role' and 'content'
        temperature (float): Controls randomness (0=deterministic, 1=random)
        max_tokens (int): Maximum response length

    Returns:
        dict: API response or error information
    """

    api_key = os.environ.get('OPENROUTER_API_KEY')
    if not api_key:
        return {"error": "API key not set. Please run the setup cell above."}

    url = "https://openrouter.ai/api/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {api_key}",
        "HTTP-Referer": "https://github.com/adedamolaobadeji",
        "X-Title": "LLM Training Notebook",
        "Content-Type": "application/json"
    }

    payload = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    try:
        response = requests.post(url, json=payload, headers=headers, timeout=30)
        return response.json()
    except requests.exceptions.Timeout:
        return {"error": "Request timed out"}
    except requests.exceptions.RequestException as e:
        return {"error": f"Request failed: {str(e)}"}

print("✓ API helper function created!")
print("\nFunction: call_openrouter_api()")
print("  - Handles authentication")
print("  - Manages HTTP requests")
print("  - Returns JSON responses")

In [ ]:
# Helper function to extract response text

def extract_response(api_response):
    """
    Extract the actual text response from API response.

    Args:
        api_response (dict): Response from OpenRouter API

    Returns:
        str: Extracted text or error message
    """

    if "error" in api_response:
        return f"❌ Error: {api_response['error']}"

    if "choices" in api_response and len(api_response["choices"]) > 0:
        return api_response["choices"][0]["message"]["content"]

    return "❌ Unexpected response format"

# Helper function to display API usage stats

def display_usage(api_response):
    """
    Display token usage statistics from API response.
    """
    if "usage" in api_response:
        usage = api_response["usage"]
        print("\n📊 API Usage Statistics:")
        print(f"  • Prompt tokens: {usage.get('prompt_tokens', 'N/A')}")
        print(f"  • Completion tokens: {usage.get('completion_tokens', 'N/A')}")
        print(f"  • Total tokens: {usage.get('total_tokens', 'N/A')}")

print("✓ Helper functions created!")

---

## Section 4: Testing Different Models 🚀

Let's explore what models are available and test a few of them.

### Popular Models on OpenRouter:
- **OpenAI Models:** `openai/gpt-4`, `openai/gpt-3.5-turbo`
- **Anthropic Claude:** `anthropic/claude-3-opus`, `anthropic/claude-3-sonnet`
- **Open Source:** `mistralai/mistral-large`, `meta-llama/llama-2-70b`
- **Google:** `google/palm-2` (deprecated, use Gemini now)

In [ ]:
# Test 1: Simple API call to check authentication
print("🧪 Test 1: Checking API Connection...\n")

# Create a simple test message
test_messages = [
    {"role": "user", "content": "Say 'Hello, I'm working!' and nothing else."}
]

# Call the API with a lightweight model
response = call_openrouter_api(
    model="openai/gpt-3.5-turbo",  # Lightweight, fast, and cheap
    messages=test_messages,
    temperature=0.7,
    max_tokens=50
)

# Display results
if "error" not in response:
    print("✅ API Connection Successful!\n")
    print(f"Response: {extract_response(response)}")
    display_usage(response)
else:
    print(f"❌ Connection Failed: {response['error']}")
    print("\nTroubleshooting:")
    print("1. Check if your API key is correct")
    print("2. Verify you have API credits (free tier has limits)")
    print("3. Check your internet connection")

In [ ]:
# Test 2: Compare different models
print("\n🧪 Test 2: Testing Different Models...\n")
print("Sending the same prompt to different models...\n")

test_prompt = "Explain machine learning in 2 sentences."

models_to_test = [
    ("openai/gpt-3.5-turbo", "GPT-3.5 Turbo"),
    ("mistralai/mistral-small", "Mistral Small"),
]

results = []

for model_id, model_name in models_to_test:
    print(f"Testing {model_name}...")

    response = call_openrouter_api(
        model=model_id,
        messages=[{"role": "user", "content": test_prompt}],
        temperature=0.7,
        max_tokens=100
    )

    if "error" not in response:
        answer = extract_response(response)
        tokens = response.get("usage", {}).get("total_tokens", "N/A")
        results.append({
            "Model": model_name,
            "Response": answer[:100] + "..." if len(answer) > 100 else answer,
            "Tokens Used": tokens
        })
        print(f"  ✓ Success\n")
    else:
        print(f"  ✗ Error: {response['error']}\n")
        results.append({
            "Model": model_name,
            "Response": "Error",
            "Tokens Used": "N/A"
        })

# Display results in a table
if results:
    df = pd.DataFrame(results)
    print("\n📊 Model Comparison:")
    print(df.to_string(index=False))

---

## Section 5: Prompt Engineering Fundamentals 💡

### What is a Prompt?
A prompt is the input you give to an LLM to get a response. The quality of your prompt directly affects the quality of the response.

### Key Principles:

1. **Be Specific:** Clear instructions get better results
2. **Provide Context:** Give the model background information
3. **Use Examples:** Few-shot learning (showing examples) improves performance
4. **Define Output Format:** Specify how you want the response structured
5. **Set a Role:** Tell the model what role to play (e.g., "You are a Python expert")

### Prompt Structure:
```
[CONTEXT] → [TASK] → [CONSTRAINTS] → [OUTPUT FORMAT]
```

In [ ]:
print("📝 Prompt Engineering Examples\n")
print("="*60)

print("\n❌ WEAK PROMPT:")
weak_prompt = "Tell me about Python"
print(f"Prompt: '{weak_prompt}'")

response = call_openrouter_api(
    model="openai/gpt-3.5-turbo",
    messages=[{"role": "user", "content": weak_prompt}],
    temperature=0.7,
    max_tokens=200
)
print(f"\nResponse: {extract_response(response)}")

print("\n" + "="*60)
print("\n✅ STRONG PROMPT:")
strong_prompt = """You are an expert Python instructor.
Explain Python for beginners in 3-4 sentences.
Focus on: what it is, why it's popular, and one key benefit.
Use simple language."""
print(f"Prompt: '{strong_prompt}'")

response = call_openrouter_api(
    model="openai/gpt-3.5-turbo",
    messages=[{"role": "user", "content": strong_prompt}],
    temperature=0.7,
    max_tokens=200
)
print(f"\nResponse: {extract_response(response)}")

print("\n" + "="*60)
print("\n💬 Notice the difference?")
print("The strong prompt gets more focused and relevant responses!")

In [ ]:
# Example 2: Few-Shot Learning (Providing Examples)

print("\n\n🎓 Few-Shot Learning: Teaching by Example\n")
print("="*60)

few_shot_prompt = """Convert user requests into structured JSON.

Examples:
User: "I want a red car"
JSON: {"item": "car", "color": "red"}

User: "Get me a large coffee"
JSON: {"item": "coffee", "size": "large"}

Now convert this:
User: "I need a small blue notebook"""

response = call_openrouter_api(
    model="openai/gpt-3.5-turbo",
    messages=[{"role": "user", "content": few_shot_prompt}],
    temperature=0.3,  # Lower temperature for more consistent outputs
    max_tokens=100
)

print(f"Prompt:\n{few_shot_prompt}")
print(f"\n\nResponse:\n{extract_response(response)}")
print("\n" + "="*60)
print("\n💡 By providing examples, the model learns the exact format you want!")

In [ ]:
# Example 3: Role-Playing (Setting Context)

print("\n\n🎭 Role-Playing: Setting the Model's Persona\n")
print("="*60)

role_prompt = """You are a sarcastic tech support representative.
A customer asks: "Why is my computer slow?"
Respond in a sarcastic but helpful way (max 3 sentences)."""

response = call_openrouter_api(
    model="openai/gpt-3.5-turbo",
    messages=[{"role": "user", "content": role_prompt}],
    temperature=0.8,  # Higher for more creative/varied responses
    max_tokens=150
)

print(f"Prompt: {role_prompt}")
print(f"\nResponse:\n{extract_response(response)}")

print("\n" + "="*60)
print("\n💡 Assigning a role makes the model respond in that style!")

---

## Section 6: Experimenting with Parameters ⚙️

LLMs have several parameters that control their behavior. Let's explore the most important ones.

### Key Parameters:

**1. `temperature` (0.0 to 2.0)**
- Controls randomness/creativity
- 0.0 = Always same output (deterministic)
- 1.0 = Balanced (default)
- 2.0 = Very random/creative
- **Use:** Lower for facts, higher for creative writing

**2. `max_tokens`**
- Maximum length of response
- 1 token ≈ 4 characters
- Limits response length and cost
- **Use:** Set based on expected answer length

**3. `top_p` (nucleus sampling)**
- Controls diversity (alternative to temperature)
- Only consider tokens with cumulative probability up to this value
- 1.0 = Consider all tokens (default)
- **Use:** Can combine with temperature for fine control

In [ ]:
# Experiment 1: Temperature Effects

print("🔬 Temperature Experiment: How does it affect creativity?\n")
print("="*70)

temperature_prompt = "Complete this sentence: 'The future of AI is...'"

temperatures = [0.1, 0.7, 1.5]

for temp in temperatures:
    print(f"\n📊 Temperature = {temp}")
    print("-" * 70)

    # Get 2 responses to show variability
    for i in range(2):
        response = call_openrouter_api(
            model="openai/gpt-3.5-turbo",
            messages=[{"role": "user", "content": temperature_prompt}],
            temperature=temp,
            max_tokens=50
        )
        print(f"Response {i+1}: {extract_response(response)}")

print("\n" + "="*70)
print("\n💡 Observations:")
print("• Low temperature (0.1): More consistent, similar responses")
print("• Medium temperature (0.7): Balanced, varied but logical")
print("• High temperature (1.5): Very creative, unpredictable")

In [ ]:
# Experiment 2: Max Tokens Effect

print("\n\n🔬 Max Tokens Experiment: Controlling Response Length\n")
print("="*70)

length_prompt = "What is blockchain technology? Explain in detail."

token_limits = [50, 150, 300]

for max_tok in token_limits:
    print(f"\n📝 Max Tokens = {max_tok}")
    print("-" * 70)

    response = call_openrouter_api(
        model="openai/gpt-3.5-turbo",
        messages=[{"role": "user", "content": length_prompt}],
        temperature=0.7,
        max_tokens=max_tok
    )

    answer = extract_response(response)
    actual_tokens = response.get("usage", {}).get("completion_tokens", "N/A")

    print(f"Response (truncated):")
    print(answer[:150] + "..." if len(answer) > 150 else answer)
    print(f"\nActual tokens used: {actual_tokens}")

print("\n" + "="*70)
print("\n💡 Key Insight:")
print("max_tokens caps the response length. Longer responses = higher cost!")

In [ ]:
# Experiment 3: Parameter Comparison

print("\n\n🔬 Side-by-Side Parameter Comparison\n")
print("="*70)

comparison_prompt = "Write a creative product name for a coffee shop AI app"

configs = [
    {"temp": 0.3, "max_tok": 30, "label": "Conservative"},
    {"temp": 0.7, "max_tok": 50, "label": "Balanced"},
    {"temp": 1.2, "max_tok": 75, "label": "Creative"},
]

for config in configs:
    print(f"\n🎯 {config['label']} (temp={config['temp']}, max_tokens={config['max_tok']})")
    print("-" * 70)

    response = call_openrouter_api(
        model="openai/gpt-3.5-turbo",
        messages=[{"role": "user", "content": comparison_prompt}],
        temperature=config['temp'],
        max_tokens=config['max_tok']
    )

    print(f"Response: {extract_response(response)}")
    display_usage(response)

print("\n" + "="*70)

---

## Section 7: Handling Errors and Rate Limits 🛡️

When working with APIs, things can go wrong. Let's learn how to handle errors gracefully.

### Common Errors:

1. **401 Unauthorized** - Invalid API key
2. **429 Too Many Requests** - Rate limit exceeded
3. **500 Server Error** - OpenRouter server issue
4. **Timeout** - Request took too long

### Best Practices:
- Always wrap API calls in try-except blocks
- Implement retry logic with exponential backoff
- Check rate limit headers
- Log errors for debugging
- Set reasonable timeouts

In [ ]:
# Robust API call with error handling and retry logic

def call_openrouter_with_retry(model, messages, temperature=0.7, max_tokens=500, max_retries=3):
    """
    Call OpenRouter API with retry logic and comprehensive error handling.

    Args:
        model (str): Model ID
        messages (list): Message list
        temperature (float): Temperature parameter
        max_tokens (int): Max tokens in response
        max_retries (int): Number of retry attempts

    Returns:
        dict: API response or error details
    """

    api_key = os.environ.get('OPENROUTER_API_KEY')
    if not api_key:
        return {
            "error": "API key not found",
            "details": "Set OPENROUTER_API_KEY environment variable"
        }

    url = "https://openrouter.ai/api/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {api_key}",
        "HTTP-Referer": "https://github.com/adedamolaobadeji",
        "X-Title": "LLM Training Notebook",
        "Content-Type": "application/json"
    }

    payload = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    for attempt in range(max_retries):
        try:
            response = requests.post(url, json=payload, headers=headers, timeout=30)

            # Check for rate limiting
            if response.status_code == 429:
                wait_time = int(response.headers.get('Retry-After', 5))
                print(f"⚠️  Rate limited. Waiting {wait_time} seconds... (Attempt {attempt+1}/{max_retries})")
                time.sleep(wait_time)
                continue

            # Check for authentication errors
            if response.status_code == 401:
                return {
                    "error": "Authentication failed",
                    "details": "Invalid API key. Check your credentials."
                }

            # Check for server errors
            if response.status_code >= 500:
                if attempt < max_retries - 1:
                    wait_time = 2 ** attempt  # Exponential backoff
                    print(f"⚠️  Server error. Retrying in {wait_time} seconds... (Attempt {attempt+1}/{max_retries})")
                    time.sleep(wait_time)
                    continue

            # Success
            response.raise_for_status()
            return response.json()

        except requests.exceptions.Timeout:
            if attempt < max_retries - 1:
                print(f"⚠️  Request timeout. Retrying... (Attempt {attempt+1}/{max_retries})")
                time.sleep(1)
            else:
                return {"error": "Request timeout after retries"}

        except requests.exceptions.RequestException as e:
            return {"error": f"Request failed: {str(e)}"}

    return {"error": "Max retries exceeded"}

print("✓ Robust API function with error handling created!")

In [ ]:
# Test the error handling

print("\n🧪 Testing Error Handling\n")
print("="*70)

print("\n✅ Test 1: Valid Request")
response = call_openrouter_with_retry(
    model="openai/gpt-3.5-turbo",
    messages=[{"role": "user", "content": "Say 'Success!' briefly"}],
    max_tokens=50
)

if "error" not in response:
    print(f"Response: {extract_response(response)}")
else:
    print(f"Error: {response['error']}")

print("\n" + "="*70)

print("\n❌ Test 2: Invalid API Key (Simulated)")
bad_key = os.environ['OPENROUTER_API_KEY']
os.environ['OPENROUTER_API_KEY'] = "invalid-key"

response = call_openrouter_with_retry(
    model="openai/gpt-3.5-turbo",
    messages=[{"role": "user", "content": "Test"}],
    max_tokens=50
)

if "error" in response:
    print(f"Error caught: {response['error']}")
    print(f"Details: {response.get('details', 'N/A')}")

# Restore the key
os.environ['OPENROUTER_API_KEY'] = bad_key

print("\n" + "="*70)

---

## Section 8: Advanced Use Cases 🚀

Now let's explore real-world applications of LLMs!

### Use Cases We'll Cover:
1. **Text Summarization** - Condense long text
2. **Translation** - Translate between languages
3. **Code Generation** - Write code from descriptions
4. **Content Analysis** - Extract insights from text
5. **Conversational AI** - Multi-turn conversations

In [ ]:
# Use Case 1: Text Summarization

print("\n📚 Use Case 1: Text Summarization\n")
print("="*70)

long_text = """Machine learning is a subset of artificial intelligence that enables
systems to learn and improve from experience without being explicitly programmed.
It focuses on developing computer programs that can access data and use it to
learn for themselves. The process of learning begins with observations or data,
such as examples, direct experience, or instruction, to look for patterns in
data and make better decisions in the future based on the examples provided.
The primary aim is to allow computers to learn automatically without human
intervention or assistance and adjust actions accordingly."""

summarization_prompt = f"""Summarize this text in 1-2 sentences:

{long_text}

Summary:"""

response = call_openrouter_with_retry(
    model="openai/gpt-3.5-turbo",
    messages=[{"role": "user", "content": summarization_prompt}],
    temperature=0.3,
    max_tokens=100
)

print("Original text:")
print(f"{long_text}\n")
print("-" * 70)
print("\nSummary:")
print(extract_response(response))
print("\n" + "="*70)

In [ ]:
# Use Case 2: Translation

print("\n🌍 Use Case 2: Language Translation\n")
print("="*70)

english_text = "Hello! How are you today? I hope you're having a great day."

translation_prompt = f"""Translate this English text to Spanish:

'{english_text}'

Spanish translation:"""

response = call_openrouter_with_retry(
    model="openai/gpt-3.5-turbo",
    messages=[{"role": "user", "content": translation_prompt}],
    temperature=0.3,
    max_tokens=100
)

print(f"English: {english_text}\n")
print("-" * 70)
print(f"\nSpanish: {extract_response(response)}")
print("\n" + "="*70)

In [ ]:
# Use Case 3: Code Generation

print("\n💻 Use Case 3: Code Generation\n")
print("="*70)

code_prompt = """Write a Python function that:
1. Takes a list of numbers as input
2. Returns the sum of all even numbers
3. Includes error handling

Provide only the code, no explanation."""

response = call_openrouter_with_retry(
    model="openai/gpt-3.5-turbo",
    messages=[{"role": "user", "content": code_prompt}],
    temperature=0.3,
    max_tokens=250
)

print(f"Request: {code_prompt}\n")
print("-" * 70)
print("\nGenerated Code:")
print(extract_response(response))
print("\n" + "="*70)

In [ ]:
# Use Case 4: Multi-turn Conversation

print("\n💬 Use Case 4: Multi-Turn Conversation\n")
print("="*70)

print("Let's have a conversation with the AI!\n")

conversation_history = [
    {"role": "user", "content": "What's your favorite programming language?"}
]

print("User: What's your favorite programming language?")

response = call_openrouter_with_retry(
    model="openai/gpt-3.5-turbo",
    messages=conversation_history,
    temperature=0.7,
    max_tokens=150
)

assistant_response = extract_response(response)
print(f"\nAssistant: {assistant_response}")

# Add assistant response to history
conversation_history.append({"role": "assistant", "content": assistant_response})

# Continue conversation
conversation_history.append({"role": "user", "content": "Why do you like that language?"})

print("\n" + "-" * 70)
print("\nUser: Why do you like that language?")

response = call_openrouter_with_retry(
    model="openai/gpt-3.5-turbo",
    messages=conversation_history,
    temperature=0.7,
    max_tokens=150
)

assistant_response = extract_response(response)
print(f"\nAssistant: {assistant_response}")

print("\n" + "="*70)
print("\n💡 Notice: The AI remembers the conversation context!")
print("This is how chatbots maintain coherent conversations.")

In [ ]:
# Use Case 5: Content Analysis & Sentiment

print("\n📊 Use Case 5: Sentiment Analysis & Content Insights\n")
print("="*70)

reviews = [
    "This product is amazing! Best purchase I've made.",
    "Terrible quality. Broke after one week. Very disappointed.",
    "It's okay. Does what it's supposed to do."
]

analysis_prompt = """Analyze the sentiment of these reviews. For each, provide:
1. Sentiment (Positive/Negative/Neutral)
2. Score (1-10)
3. Key emotion

Format as: Review | Sentiment | Score | Key Emotion

Reviews:
{}

Analysis:""".format("\n".join(reviews))

response = call_openrouter_with_retry(
    model="openai/gpt-3.5-turbo",
    messages=[{"role": "user", "content": analysis_prompt}],
    temperature=0.3,
    max_tokens=300
)

print("Reviews to analyze:")
for i, review in enumerate(reviews, 1):
    print(f"{i}. {review}")

print("\n" + "-" * 70)
print("\nAnalysis Results:")
print(extract_response(response))
print("\n" + "="*70)

---

## 🎓 Summary & Next Steps

### What You've Learned:

✅ **Setup & Authentication**
- How to get and set up an OpenRouter API key
- How to authenticate requests

✅ **API Fundamentals**
- Structure of API requests and responses
- Different endpoints and models

✅ **Prompt Engineering**
- Writing effective prompts
- Few-shot learning
- Role-playing and context

✅ **Parameter Tuning**
- Temperature for creativity control
- Max tokens for response length
- Impact on cost and quality

✅ **Error Handling**
- Catching and managing errors
- Retry logic with exponential backoff
- Rate limit handling

✅ **Real-World Applications**
- Summarization
- Translation
- Code generation
- Conversational AI
- Content analysis

### Recommended Next Steps:

1. **Experiment More:** Try different models and parameters
2. **Build Projects:** Create a chatbot, summarizer, or translator
3. **Optimize Costs:** Monitor token usage and choose efficient models
4. **Explore Advanced Features:** Function calling, streaming, embeddings
5. **Learn Best Practices:** Read OpenRouter and OpenAI documentation

### Resources:

- 🔗 [OpenRouter Documentation](https://openrouter.ai/docs)
- 🔗 [OpenAI API Documentation](https://platform.openai.com/docs)
- 🔗 [Prompt Engineering Guide](https://github.com/brexhq/prompt-engineering)
- 🔗 [LLM Best Practices](https://github.com/openai/openai-cookbook)

---

## 🙋 Questions?

Feel free to experiment with the code above! Try:
- Different models
- New prompts
- Different parameter combinations
- Building your own use cases

**Happy learning! 🚀**

In [ ]:
# Bonus: Create a simple CLI for testing

print("\n" + "="*70)
print("🎮 INTERACTIVE PLAYGROUND")
print("="*70)
print("""
You can now test different models and prompts interactively!

Example:
```python
response = call_openrouter_with_retry(
    model="openai/gpt-3.5-turbo",
    messages=[{"role": "user", "content": "Your prompt here"}],
    temperature=0.7,
    max_tokens=200
)
print(extract_response(response))
```

Try it in a cell below!
""")

# Store available models for reference
available_models = {
    "Fast & Cheap": [
        "openai/gpt-3.5-turbo",
        "mistralai/mistral-small",
    ],
    "Balanced": [
        "openai/gpt-4-turbo",
        "anthropic/claude-3-sonnet",
    ],
    "Powerful": [
        "openai/gpt-4",
        "anthropic/claude-3-opus",
    ]
}

print("\n📋 Available Models by Category:\n")
for category, models in available_models.items():
    print(f"{category}:")
    for model in models:
        print(f"  • {model}")
    print()